# EDA — World Health Organization (WHO)

Notebook phân tích file `who.json`, tập trung vào **domain**, thời gian, chất lượng dữ liệu và độ dài nội dung.

WHO có 8 domain trang cha. Vì output bị giới hạn theo số bài mới nhất, một số domain có thể chưa xuất hiện trong quota hiện tại; bảng coverage sẽ chỉ rõ.

In [ ]:
from pathlib import Path
import json

SOURCE_KEY = "who"
SOURCE_TITLE = "World Health Organization (WHO)"
EXPECTED_DOMAINS = [
    'News',
    'Emergencies',
    'Campaigns',
    'Events',
    'Statements',
    'Feature stories',
    'Speeches',
    'Commentaries'
]

# Tìm file dù notebook được mở từ project root, loc_crawl/ hoặc notebooks/.
roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
candidates = []
for root in roots:
    candidates.extend([
        root / "output" / f"{SOURCE_KEY}.json",
        root / "loc_crawl" / "output" / f"{SOURCE_KEY}.json",
    ])
DATA_PATH = next((path for path in dict.fromkeys(candidates) if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Không tìm thấy {SOURCE_KEY}.json trong output/ hoặc loc_crawl/output/")

records = json.loads(DATA_PATH.read_text(encoding="utf-8"))
print(f"Nguồn: {SOURCE_TITLE}")
print(f"File: {DATA_PATH}")
print(f"Số bản ghi: {len(records):,}")

## 1. Chuẩn hóa trường phục vụ EDA

Không sửa file JSON gốc. Các cột bên dưới chỉ tồn tại trong DataFrame của notebook.

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
df = pd.DataFrame(records)
preview_columns = [column for column in ["publish_date", "domain", "claim", "url"] if column in df]
display(df[preview_columns].head(3))

PUBLIC_FIELDS = [
    "id", "source_type", "source_name", "url", "domain", "publish_date",
    "claim", "original_text", "label", "evidence", "justification", "comments",
]
for column in PUBLIC_FIELDS:
    if column not in df.columns:
        df[column] = pd.NA

for column in ["domain", "claim", "original_text", "justification", "url", "id"]:
    df[column] = df[column].fillna("").astype(str)

df["domain_clean"] = df["domain"].str.strip()
df["publish_dt"] = pd.to_datetime(df["publish_date"], errors="coerce")
df["year"] = df["publish_dt"].dt.year.astype("Int64")
df["domain_tokens"] = df["domain_clean"].apply(
    lambda value: [token.strip() for token in value.split(";") if token.strip()]
)
df["domain_count"] = df["domain_tokens"].str.len()
for column in ["claim", "original_text", "justification"]:
    df[f"{column}_len"] = df[column].str.len()

# Mỗi dòng trong domain_long tương ứng một bài × một domain đơn.
domain_long = df.explode("domain_tokens").rename(columns={"domain_tokens": "domain_atomic"})
domain_long["domain_atomic"] = domain_long["domain_atomic"].fillna("").str.strip()
domain_long = domain_long[domain_long["domain_atomic"].ne("")].copy()

overview = pd.Series({
    "records": len(df),
    "columns": len(df.columns),
    "unique_urls": df["url"].nunique(),
    "unique_domain_strings": df["domain_clean"].nunique(),
    "atomic_domains": domain_long["domain_atomic"].nunique(),
    "min_date": df["publish_dt"].min(),
    "max_date": df["publish_dt"].max(),
}, name="value").to_frame()
display(overview)

## 2. Phân bố domain và độ phủ taxonomy

In [ ]:
domain_summary = (
    domain_long.groupby("domain_atomic", dropna=False)
    .agg(articles=("url", "nunique"))
    .sort_values("articles", ascending=False)
)
domain_summary["share_of_records_pct"] = (
    domain_summary["articles"].div(len(df)).mul(100).round(2)
)

observed = set(domain_summary.index)
coverage = pd.DataFrame({
    "domain": EXPECTED_DOMAINS,
    "present": [domain in observed for domain in EXPECTED_DOMAINS],
    "articles": [int(domain_summary["articles"].get(domain, 0)) for domain in EXPECTED_DOMAINS],
})
coverage["share_pct"] = coverage["articles"].div(len(df)).mul(100).round(2)
unexpected = sorted(observed - set(EXPECTED_DOMAINS))

print("Bảng tần suất domain đơn:")
display(domain_summary)
print("Độ phủ taxonomy mong đợi:")
display(coverage)
print("Domain ngoài danh sách mong đợi:", unexpected or "Không có")

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

ax = domain_summary.sort_values("articles")["articles"].plot(
    kind="barh", figsize=(11, max(4, 0.48 * len(domain_summary))), color="#2878B5"
)
ax.set_title(f"{SOURCE_TITLE} — số bài theo domain")
ax.set_xlabel("Số bài (một bài có thể được tính ở nhiều domain)")
ax.set_ylabel("Domain")
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=9)
plt.tight_layout()
plt.show()

## 3. Domain theo năm

Bảng và biểu đồ dùng `publish_date`. Ngày không parse được được thống kê riêng ở phần chất lượng dữ liệu.

In [ ]:
year_domain = (
    domain_long.dropna(subset=["year"])
    .groupby(["year", "domain_atomic"])["url"]
    .nunique()
    .unstack(fill_value=0)
    .sort_index()
)

if year_domain.empty:
    print("Không có publish_date hợp lệ để phân tích theo năm.")
else:
    display(year_domain)
    ax = year_domain.plot(kind="bar", stacked=True, figsize=(13, 6), colormap="tab20")
    ax.set_title(f"{SOURCE_TITLE} — cơ cấu domain theo năm")
    ax.set_xlabel("Năm")
    ax.set_ylabel("Số bài")
    ax.legend(title="Domain", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## 4. Bài đa-domain và tổ hợp domain

In [ ]:
multi_summary = (
    df["domain_count"].value_counts(dropna=False).sort_index().rename("articles").to_frame()
)
multi_summary.index.name = "number_of_domains"
display(multi_summary)

combination_summary = (
    df.assign(domain_combination=df["domain_tokens"].apply(lambda values: "; ".join(values)))
    .groupby("domain_combination")
    .size()
    .sort_values(ascending=False)
    .rename("articles")
    .head(25)
    .to_frame()
)
print("25 tổ hợp domain phổ biến nhất:")
display(combination_summary)

ax = multi_summary["articles"].plot(kind="bar", figsize=(8, 4), color="#F28E2B", rot=0)
ax.set_title(f"{SOURCE_TITLE} — số domain trên mỗi bài")
ax.set_xlabel("Số domain")
ax.set_ylabel("Số bài")
plt.tight_layout()
plt.show()

## 5. Chất lượng dữ liệu

In [ ]:
quality = pd.Series({
    "rows": len(df),
    "duplicate_url_rows": int(df.duplicated("url", keep=False).sum()),
    "duplicate_id_rows": int(df.duplicated("id", keep=False).sum()),
    "blank_domain": int(df["domain_clean"].eq("").sum()),
    "invalid_or_blank_date": int(df["publish_dt"].isna().sum()),
    "blank_claim": int(df["claim"].str.strip().eq("").sum()),
    "blank_original_text": int(df["original_text"].str.strip().eq("").sum()),
    "blank_justification": int(df["justification"].str.strip().eq("").sum()),
}, name="count").to_frame()
quality["pct"] = quality["count"].div(len(df)).mul(100).round(2)
display(quality)

missing = df[PUBLIC_FIELDS].isna().sum().to_frame("missing")
missing["blank_string"] = [
    int(df[column].astype(str).str.strip().isin(["", "nan", "None", "<NA>"]).sum())
    for column in PUBLIC_FIELDS
]
missing["missing_or_blank_pct"] = (
    (missing["missing"] + missing["blank_string"]).clip(upper=len(df)).div(len(df)).mul(100).round(2)
)
display(missing.sort_values("missing_or_blank_pct", ascending=False))
print("Label:", df["label"].value_counts(dropna=False).to_dict())

## 6. Độ dài nội dung theo domain

In [ ]:
length_summary = (
    domain_long.groupby("domain_atomic")[["claim_len", "original_text_len", "justification_len"]]
    .agg(["count", "median", "mean", "max"])
    .round(1)
)
display(length_summary)

plot_data = domain_long.loc[domain_long["justification_len"].gt(0), ["domain_atomic", "justification_len"]]
if not plot_data.empty:
    plot_data.boxplot(
        column="justification_len", by="domain_atomic", figsize=(13, 6), rot=65,
        showfliers=False, grid=False,
    )
    plt.suptitle("")
    plt.title(f"{SOURCE_TITLE} — phân bố độ dài justification theo domain")
    plt.xlabel("Domain")
    plt.ylabel("Số ký tự")
    plt.tight_layout()
    plt.show()

## 7. Xem mẫu bài theo từng domain

In [ ]:
SAMPLE_PER_DOMAIN = 2
sample_columns = ["publish_date", "domain", "claim", "url"]
for domain in domain_summary.index:
    print(f"\n### {domain}")
    sample_urls = domain_long.loc[domain_long["domain_atomic"].eq(domain), "url"].drop_duplicates().head(SAMPLE_PER_DOMAIN)
    display(df[df["url"].isin(sample_urls)][sample_columns].head(SAMPLE_PER_DOMAIN))

## 8. Xuất bảng EDA (tùy chọn)

Đổi `EXPORT_TABLES = True` nếu muốn ghi các bảng tổng hợp ra `loc_crawl/output/eda/`.

In [ ]:
EXPORT_TABLES = False
if EXPORT_TABLES:
    export_dir = DATA_PATH.parent / "eda"
    export_dir.mkdir(parents=True, exist_ok=True)
    domain_summary.to_csv(export_dir / f"{SOURCE_KEY}_domain_summary.csv", encoding="utf-8-sig")
    coverage.to_csv(export_dir / f"{SOURCE_KEY}_domain_coverage.csv", index=False, encoding="utf-8-sig")
    year_domain.to_csv(export_dir / f"{SOURCE_KEY}_domain_by_year.csv", encoding="utf-8-sig")
    quality.to_csv(export_dir / f"{SOURCE_KEY}_quality.csv", encoding="utf-8-sig")
    print("Đã xuất:", export_dir)
else:
    print("EXPORT_TABLES=False — không ghi thêm file.")